# Day 5 — **Personal AI Knowledge Worker (RAG)** — Colab‑Ready (T4 GPU)

This notebook implements the **major challenge** from Lesson 127: build your own **private Knowledge Worker** on top of your documents using **RAG** (Retrieval‑Augmented Generation).

**What you can do here**
- Point to a folder (local/Drive) or upload files → **index** everything (chunk → embed → store)
- Choose vector store (**Chroma** persistent or **FAISS** in‑memory)
- Choose embeddings (**OpenAI** or **local** Sentence‑Transformers)
- Choose LLM (**OpenAI** frontier or **open‑source** model in 4‑bit on T4)
- **Chat** with your knowledge base (with sources, prompt preview, top‑K control)

> ⚠️ **Privacy note:** If you use **OpenAI** embeddings/LLM, your prompts & snippets are sent to their API. If you need everything local, pick the **local** embeddings and **open‑source** LLM options.

## 0) Runtime check

In [1]:
import os, sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import torch
    print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch not installed yet:", e)

Python: 3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:48:23) [GCC 13.3.0]
Platform: Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39
Torch: 2.7.1 | CUDA available: True
GPU: NVIDIA GeForce RTX 3060


## 1) Installs (Colab‑friendly)

In [2]:
# # Core RAG & vector stores
# !pip install -q -U langchain langchain-community langchain-openai langchain-text-splitters
# !pip install -q -U chromadb faiss-cpu

# # Embeddings & LLMs
# !pip install -q -U sentence-transformers transformers accelerate bitsandbytes tiktoken

# Environment variables
!pip install -q -U python-dotenv

# File loaders
!pip install -q -U pypdf docx2txt python-pptx

# # UI & utils
# !pip install -q -U gradio pandas

# # Frontier clients (optional)
# !pip install -q -U openai==1.*

## 2) Imports & device

In [3]:
import os, re, io, json, glob, shutil, textwrap, tempfile, warnings
import datetime as dt
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
import numpy as np

# Environment variable loading (like day5.ipynb)
from dotenv import load_dotenv

# LangChain pieces
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma, FAISS
from langchain_openai import OpenAIEmbeddings

# Use the correct HuggingFace embeddings import (not deprecated)
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    # Fallback to the old import if the new package isn't installed
    from langchain_community.embeddings import HuggingFaceEmbeddings

# LLM (local) – we'll use transformers directly for full control
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Frontier client (optional)
try:
    from openai import OpenAI
except Exception:
    OpenAI = None

# File parsing helpers
from pypdf import PdfReader
import docx2txt
from pptx import Presentation

import gradio as gr

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

/home/hafnium/anaconda3/envs/llms/lib/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


## 3) (Optional) Google Drive for your Knowledge Base

In [4]:
# try:
#     from google.colab import drive  # type: ignore
#     drive.mount('/content/drive')
#     print("Drive mounted at /content/drive")
# except Exception:
#     print("Not running in Colab or Drive not available.")

## 4) Configuration & paths

**Environment Setup:** Create a `.env` file in your project directory with your OpenAI API key:

```bash
# Create .env file in the same directory as this notebook
echo "OPENAI_API_KEY=sk-your-actual-api-key-here" > .env
```

**Alternative:** Set environment variable directly in terminal:
```bash
export OPENAI_API_KEY="sk-your-actual-api-key-here"
```

**For WSL2/Ubuntu users:** You can also set the environment variable in your `~/.bashrc`:
```bash
echo 'export OPENAI_API_KEY="sk-your-actual-api-key-here"' >> ~/.bashrc
source ~/.bashrc
```

In [5]:
# Load environment variables from .env file (similar to day5.ipynb)
from dotenv import load_dotenv

# Load environment variables in a file called .env
load_dotenv(override=True)

# Where to persist Chroma collections by default
DEFAULT_PERSIST_DIR = "personal-knowledge-base" # "/content/chroma_kb"
os.makedirs(DEFAULT_PERSIST_DIR, exist_ok=True)

# Frontier API key (from environment or fallback)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", None)
if not OPENAI_API_KEY:
    # Fallback for manual setting (update with your actual key)
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
    
openai_client = None
if OPENAI_API_KEY and OPENAI_API_KEY != 'your-key-if-not-using-env' and OpenAI is not None:
    try:
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
        print("✅ OpenAI client ready")
    except Exception as e:
        print("⚠️ OpenAI client init failed:", e)
elif not OPENAI_API_KEY or OPENAI_API_KEY == 'your-key-if-not-using-env':
    print("⚠️ OpenAI API key not found. Please set OPENAI_API_KEY in your .env file or environment.")

# Open‑source LLM default
DEFAULT_LOCAL_LLM = "microsoft/Phi-3-mini-4k-instruct"  # 3.8B; good for T4 in 4‑bit

✅ OpenAI client ready


## 5) Simple, robust file loaders
Supported: **.txt, .md, .pdf, .docx, .pptx, .csv**

In [6]:
def load_text_file(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def load_pdf(path: str) -> str:
    try:
        reader = PdfReader(path)
        texts = []
        for page in reader.pages:
            txt = page.extract_text() or ""
            texts.append(txt)
        return "\n".join(texts)
    except Exception as e:
        return f""

def load_docx(path: str) -> str:
    try:
        return docx2txt.process(path) or ""
    except Exception:
        return ""

def load_pptx(path: str) -> str:
    try:
        prs = Presentation(path)
        texts = []
        for slide in prs.slides:
            for shape in slide.shapes:
                if hasattr(shape, "text"):
                    texts.append(shape.text)
        return "\n".join(texts)
    except Exception:
        return ""

def load_csv(path: str, max_chars: int = 100_000) -> str:
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            txt = f.read()
        # Optionally truncate extremely large CSVs to first N chars
        return txt[:max_chars]
    except Exception:
        return ""

# Expanded supported file types (including Python and other code files like day5.ipynb)
SUPPORTED_SUFFIXES = {
    ".txt", ".md", ".pdf", ".docx", ".pptx", ".csv",
    # Code files (like day5.ipynb supports various file types)
    ".py", ".js", ".html", ".css", ".json", ".xml", ".yml", ".yaml",
    # Documentation files
    ".rst", ".tex", ".log"
}

def read_any(path: str) -> str:
    ext = os.path.splitext(path)[1].lower()
    if ext in {".txt", ".md", ".py", ".js", ".html", ".css", ".json", ".xml", ".yml", ".yaml", ".rst", ".tex", ".log"}:
        return load_text_file(path)
    elif ext == ".pdf":
        return load_pdf(path)
    elif ext == ".docx":
        return load_docx(path)
    elif ext == ".pptx":
        return load_pptx(path)
    elif ext == ".csv":
        return load_csv(path)
    else:
        return ""

## 6) Chunking

In [7]:
def chunk_documents(docs: List[Dict[str, Any]], chunk_size: int = 800, chunk_overlap: int = 200):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    out_docs = []
    for d in docs:
        for chunk in splitter.split_text(d["text"]):
            out_docs.append({
                "page_content": chunk,
                "metadata": d.get("metadata", {}).copy()
            })
    return out_docs

## 7) Embeddings: OpenAI or Local (Sentence‑Transformers)

**Note:** The local embedding fallback system has been **commented out** to use OpenAI API embeddings only. If you encounter **SafeTensors** errors with local models, the system will attempt comprehensive cache clearing automatically.

**For OpenAI-only usage:** Ensure your `OPENAI_API_KEY` is set and select "OpenAI (text-embedding-3-small)" in the UI.

**Cache Issues Fix:** If you see SafeTensors/JSON parsing errors, the system now automatically clears all HuggingFace caches. You can also manually clear the cache by running the cell below.

In [8]:
@dataclass
class EmbedderCfg:
    kind: str  # "openai" | "local-minilm" | "local-bge"
    model: str

def clear_hf_cache_comprehensive():
    """Comprehensive cache clearing for corrupted HuggingFace models"""
    try:
        import shutil
        
        # Clear multiple potential cache locations
        cache_dirs = [
            os.path.expanduser("~/.cache/huggingface"),
            os.path.expanduser("~/.cache/torch/sentence_transformers"),
            os.path.expanduser("~/.cache/transformers"),
            "/tmp/huggingface",
        ]
        
        for cache_dir in cache_dirs:
            if os.path.exists(cache_dir):
                print(f"🗑️ Clearing cache directory: {cache_dir}")
                shutil.rmtree(cache_dir)
                
        print("✅ Comprehensive cache clearing completed")
        return True
        
    except Exception as e:
        print(f"⚠️ Cache clearing failed: {e}")
        return False

def clear_hf_cache(model_name: str):
    """Clear corrupted model from HuggingFace cache"""
    try:
        import shutil
        from transformers import AutoModel
        cache_dir = AutoModel.from_pretrained.__defaults__[0] if AutoModel.from_pretrained.__defaults__ else None
        if cache_dir is None:
            cache_dir = os.path.expanduser("~/.cache/huggingface/transformers")
        
        # Clear specific model cache
        model_cache_path = os.path.join(cache_dir, model_name.replace("/", "--"))
        if os.path.exists(model_cache_path):
            print(f"🗑️ Clearing corrupted cache for {model_name}")
            shutil.rmtree(model_cache_path)
            
        # Also try sentence-transformers cache
        st_cache_dir = os.path.expanduser("~/.cache/torch/sentence_transformers")
        st_model_path = os.path.join(st_cache_dir, model_name.replace("/", "_"))
        if os.path.exists(st_model_path):
            print(f"🗑️ Clearing sentence-transformers cache for {model_name}")
            shutil.rmtree(st_model_path)
            
    except Exception as e:
        print(f"⚠️ Cache clearing failed: {e}")

def create_embedder(cfg: EmbedderCfg):
    if cfg.kind == "openai":
        if not OPENAI_API_KEY or OPENAI_API_KEY == 'your-key-if-not-using-env' or OpenAIEmbeddings is None:
            raise RuntimeError("OpenAI API key not set or OpenAIEmbeddings missing.")
        return OpenAIEmbeddings(model=cfg.model)
    elif cfg.kind in ["local-minilm", "local-bge"]:
        # Handle corrupted cache issue similar to day5.ipynb approach
        print(f"🔄 Attempting to load embedding model: {cfg.model}")
        
        # Try comprehensive cache clear first if we detect cache corruption
        try:
            # Try to import and test sentence transformers directly
            import sentence_transformers
            test_model = sentence_transformers.SentenceTransformer(cfg.model)
            test_model.encode("test")  # Quick test
            print(f"✅ Direct sentence-transformers test passed for {cfg.model}")
            
        except Exception as e:
            if "safetensors" in str(e) or "JSON" in str(e) or "EOF" in str(e):
                print(f"🗑️ Detected cache corruption, clearing all caches...")
                clear_hf_cache_comprehensive()
                
                # Wait a moment for filesystem operations to complete
                import time
                time.sleep(1)
        
        # Now try with LangChain wrapper using the modern import
        try:
            # Use the modern HuggingFace embeddings import (already imported at top)
            embedder = HuggingFaceEmbeddings(
                model_name=cfg.model,
                encode_kwargs={"normalize_embeddings": True},
                model_kwargs={"device": DEVICE}
            )
            print(f"✅ Successfully loaded embedding model: {cfg.model}")
            return embedder
            
        except Exception as e:
            print(f"⚠️ Failed to load {cfg.model}: {e}")
            
            # One more comprehensive cache clear attempt
            if "safetensors" in str(e) or "JSON" in str(e) or "EOF" in str(e):
                print("🔄 Final cache clearing attempt...")
                clear_hf_cache_comprehensive()
                
                try:
                    # Import and force re-download
                    import sentence_transformers
                    embedder = HuggingFaceEmbeddings(
                        model_name=cfg.model,
                        encode_kwargs={"normalize_embeddings": True},
                        model_kwargs={"device": DEVICE}
                    )
                    print(f"✅ Successfully loaded {cfg.model} after comprehensive cache clear")
                    return embedder
                except Exception as e2:
                    print(f"⚠️ Final retry failed for {cfg.model}: {e2}")
            
            # No fallback - fail if the requested model doesn't work (as requested)
            raise RuntimeError(f"❌ Embedding model {cfg.model} failed to load. Error: {e}")
        
        # COMMENTED OUT: Multiple fallback models system (as requested in previous modification)
        # models_to_try = [
        #     cfg.model,  # Original requested model
        #     "sentence-transformers/all-MiniLM-L6-v2",  # First fallback
        #     "sentence-transformers/all-MiniLM-L12-v2",  # Second fallback
        #     "sentence-transformers/paraphrase-MiniLM-L6-v2",  # Third fallback
        # ]
        # 
        # for attempt, model_name in enumerate(models_to_try):
        #     try:
        #         print(f"🔄 Attempting to load embedding model: {model_name}")
        #         embedder = HuggingFaceEmbeddings(
        #             model_name=model_name,
        #             encode_kwargs={"normalize_embeddings": True},
        #             model_kwargs={"device": DEVICE}
        #         )
        #         print(f"✅ Successfully loaded embedding model: {model_name}")
        #         return embedder
        #         
        #     except Exception as e:
        #         print(f"⚠️ Failed to load {model_name}: {e}")
        #         
        #         # If it's a safetensors error, try to clear cache
        #         if "safetensors" in str(e) or "JSON" in str(e):
        #             clear_hf_cache(model_name)
        #             
        #             # Try once more after cache clear
        #             try:
        #                 print(f"🔄 Retrying {model_name} after cache clear...")
        #                 embedder = HuggingFaceEmbeddings(
        #                     model_name=model_name,
        #                     encode_kwargs={"normalize_embeddings": True},
        #                     model_kwargs={"device": DEVICE}
        #                 )
        #                 print(f"✅ Successfully loaded {model_name} after cache clear")
        #                 return embedder
        #             except Exception as e2:
        #                 print(f"⚠️ Retry failed for {model_name}: {e2}")
        #         
        #         # Continue to next model if this one failed
        #         if attempt < len(models_to_try) - 1:
        #             print(f"🔄 Trying next fallback model...")
        #             continue
        # 
        # # If all models failed, raise the last error
        # raise RuntimeError(f"❌ All embedding models failed. Last error: {e}")
    else:
        raise ValueError("Unknown embedder kind")

In [9]:
# Manual Cache Clearing (run this if you encounter SafeTensors errors)
def manual_cache_clear():
    """Manually clear all HuggingFace and sentence-transformers caches"""
    import shutil
    
    cache_locations = [
        os.path.expanduser("~/.cache/huggingface"),
        os.path.expanduser("~/.cache/torch/sentence_transformers"),
        os.path.expanduser("~/.cache/transformers"),
        "/tmp/huggingface"
    ]
    
    cleared = []
    for cache_dir in cache_locations:
        try:
            if os.path.exists(cache_dir):
                shutil.rmtree(cache_dir)
                cleared.append(cache_dir)
                print(f"✅ Cleared: {cache_dir}")
            else:
                print(f"⏩ Not found: {cache_dir}")
        except Exception as e:
            print(f"⚠️ Failed to clear {cache_dir}: {e}")
    
    if cleared:
        print(f"\n🎉 Successfully cleared {len(cleared)} cache directories")
        print("💡 Restart your notebook kernel after clearing cache for best results")
    else:
        print("ℹ️ No cache directories found to clear")

# Uncomment the line below to manually clear caches if needed
# manual_cache_clear()

### Test OpenAI API Embedding

Test to verify that OpenAI API embeddings are working correctly.

In [10]:
# Test OpenAI API Embedding
def test_openai_embedding():
    """Test OpenAI API embedding functionality"""
    print("🧪 Testing OpenAI API Embedding...")
    
    # Check if OpenAI API key is available
    if not OPENAI_API_KEY:
        print("❌ OPENAI_API_KEY not found in environment variables")
        print("   Please set your OpenAI API key: export OPENAI_API_KEY='sk-...'")
        return False
    
    if openai_client is None:
        print("❌ OpenAI client not initialized")
        return False
    
    print(f"✅ OpenAI API key found: {OPENAI_API_KEY[:10]}...")
    
    try:
        # Create OpenAI embedder configuration
        openai_cfg = EmbedderCfg(kind="openai", model="text-embedding-3-small")
        print(f"📝 Testing with model: {openai_cfg.model}")
        
        # Create the embedder
        embedder = create_embedder(openai_cfg)
        print("✅ OpenAI embedder created successfully")
        
        # Test with sample text
        test_texts = [
            "This is a test document about artificial intelligence.",
            "Python is a popular programming language for data science.",
            "Machine learning models require large amounts of training data."
        ]
        
        print(f"🔄 Generating embeddings for {len(test_texts)} test documents...")
        
        # Generate embeddings
        embeddings = embedder.embed_documents(test_texts)
        
        print(f"✅ Successfully generated embeddings!")
        print(f"   - Number of embeddings: {len(embeddings)}")
        print(f"   - Embedding dimension: {len(embeddings[0]) if embeddings else 'N/A'}")
        print(f"   - Sample embedding (first 5 values): {embeddings[0][:5] if embeddings else 'N/A'}")
        
        # Test query embedding
        query = "What is machine learning?"
        query_embedding = embedder.embed_query(query)
        print(f"✅ Query embedding generated successfully!")
        print(f"   - Query: '{query}'")
        print(f"   - Query embedding dimension: {len(query_embedding)}")
        print(f"   - Query embedding (first 5 values): {query_embedding[:5]}")
        
        print("🎉 OpenAI API embedding test completed successfully!")
        return True
        
    except Exception as e:
        print(f"❌ OpenAI API embedding test failed: {e}")
        print(f"   Error type: {type(e).__name__}")
        
        # Provide specific guidance based on error type
        if "api_key" in str(e).lower():
            print("   💡 Suggestion: Check that your OpenAI API key is valid and has sufficient credits")
        elif "rate" in str(e).lower() or "quota" in str(e).lower():
            print("   💡 Suggestion: You may have hit rate limits or quota limits. Wait a moment and try again")
        elif "model" in str(e).lower():
            print("   💡 Suggestion: The embedding model may not be available. Try 'text-embedding-ada-002' instead")
        else:
            print("   💡 Suggestion: Check your internet connection and OpenAI service status")
        
        return False

# Run the test
if __name__ == "__main__":
    test_result = test_openai_embedding()
    if test_result:
        print("\n🚀 You're ready to use OpenAI embeddings for your RAG system!")
    else:
        print("\n⚠️  Please fix the issues above before using OpenAI embeddings.")
else:
    # When running in notebook, execute immediately
    test_result = test_openai_embedding()
    if test_result:
        print("\n🚀 You're ready to use OpenAI embeddings for your RAG system!")
    else:
        print("\n⚠️  Please fix the issues above before using OpenAI embeddings.")

🧪 Testing OpenAI API Embedding...
✅ OpenAI API key found: sk-proj-Zt...
📝 Testing with model: text-embedding-3-small
✅ OpenAI embedder created successfully
🔄 Generating embeddings for 3 test documents...
✅ Successfully generated embeddings!
   - Number of embeddings: 3
   - Embedding dimension: 1536
   - Sample embedding (first 5 values): [-0.005340373143553734, 0.017985880374908447, 0.023694442585110664, -0.026457490399479866, 0.011593072675168514]
✅ Query embedding generated successfully!
   - Query: 'What is machine learning?'
   - Query embedding dimension: 1536
   - Query embedding (first 5 values): [-0.002476818859577179, -0.012755980715155602, -0.006645360495895147, -0.03157883137464523, 0.028759293258190155]
🎉 OpenAI API embedding test completed successfully!

🚀 You're ready to use OpenAI embeddings for your RAG system!


## 8) Vector stores: **Chroma** (persistent) or **FAISS** (in‑memory, savable)

In [11]:
@dataclass
class VSHandle:
    kind: str  # "chroma" | "faiss"
    handle: Any

def build_vector_store(docs, embedder, kind: str = "chroma", persist_dir: str = DEFAULT_PERSIST_DIR, collection_name: str = "personal_kb"):
    if kind == "chroma":
        vs = Chroma.from_documents(
            documents=[type("Doc", (), d) for d in docs],  # quick struct with page_content/metadata
            embedding=embedder,
            persist_directory=persist_dir,
            collection_name=collection_name
        )
        return VSHandle(kind="chroma", handle=vs)
    elif kind == "faiss":
        vs = FAISS.from_documents(
            documents=[type("Doc", (), d) for d in docs],
            embedding=embedder
        )
        return VSHandle(kind="faiss", handle=vs)
    else:
        raise ValueError("Unknown vector store kind")

def save_faiss(vs_handle: VSHandle, path: str):
    assert vs_handle.kind == "faiss"
    vs_handle.handle.save_local(path)

def load_faiss(path: str, embedder) -> VSHandle:
    vs = FAISS.load_local(path, embeddings=embedder, allow_dangerous_deserialization=True)
    return VSHandle(kind="faiss", handle=vs)

## 9) Open‑source LLM loader (4‑bit on T4)

In [12]:
_tok = None
_mdl = None

def load_local_llm(model_id: str = DEFAULT_LOCAL_LLM):
    global _tok, _mdl
    if _mdl is not None and getattr(_mdl, "name_or_path", None) == model_id:
        return _tok, _mdl
    print(f"Loading local LLM: {model_id}")
    kwargs = {}
    if DEVICE == "cuda":
        try:
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
            kwargs["device_map"] = "auto"
            kwargs["torch_dtype"] = torch.bfloat16
        except Exception as e:
            print("4-bit load failed; falling back:", e)
            kwargs["device_map"] = "auto"
    _tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if _tok.pad_token is None:
        _tok.pad_token = _tok.eos_token
    _mdl = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    return _tok, _mdl

## 10) RAG core: retrieval → prompt assembly → generation

In [13]:
SYSTEM_PROMPT = (
        "You are a helpful, concise assistant. Answer using only the provided context. "
        "If the answer is not in the context, say you don't know."
        "Cite sources with (Source: filename or path)."
    )

def build_context_str(docs: List[Dict[str, Any]], max_chars: int = 8_000) -> Tuple[str, List[Dict[str, Any]]]:
    pieces = []
    used = []
    total = 0
    for d in docs:
        meta = d.get("metadata", {})
        fname = meta.get("source", meta.get("path", "unknown"))
        snippet = d.get("page_content", "")
        # limit per doc
        snippet = snippet[:1500]
        piece = f"""[Source: {fname}]
{snippet}"""
        if total + len(piece) > max_chars:
            break
        pieces.append(piece)
        used.append({"source": fname, "snippet": snippet})
        total += len(piece)
    return "\n\n".join(pieces), used

def retrieve_docs(vs_handle: VSHandle, query: str, k: int = 8):
    retriever = vs_handle.handle.as_retriever(search_kwargs={"k": int(k)})
    docs = retriever.get_relevant_documents(query)
    # Convert LC Document -> dict
    out = []
    for d in docs:
        out.append({"page_content": d.page_content, "metadata": dict(d.metadata or {})})
    return out

def generate_frontier_answer(question: str, context: str, model: str = "gpt-4o-mini", temperature: float = 0.2, max_tokens: int = 512) -> str:
    if openai_client is None:
        raise RuntimeError("OpenAI client not configured (set OPENAI_API_KEY).")
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    try:
        resp = openai_client.responses.create(
            model=model,
            input=messages,
            temperature=temperature,
            max_output_tokens=max_tokens,
        )
        return resp.output_text.strip()
    except Exception:
        chat = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return chat.choices[0].message.content.strip()

def generate_local_answer(question: str, context: str, model_id: str = DEFAULT_LOCAL_LLM, temperature: float = 0.2, max_new_tokens: int = 512) -> str:
    tok, mdl = load_local_llm(model_id=model_id)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    try:
        input_ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    except Exception:
        # Fallback: simple concatenation
        prompt = f"""{SYSTEM_PROMPT}

Context:
{context}

Question: {question}
Answer:"""
        input_ids = tok.encode(prompt, return_tensors="pt")
    if DEVICE == "cuda":
        input_ids = input_ids.to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            input_ids=input_ids,
            max_new_tokens=int(max_new_tokens),
            temperature=float(temperature),
            do_sample=True,
            top_p=0.95,
            pad_token_id=tok.eos_token_id,
        )
    gen = out[0, input_ids.shape[1]:]
    text = tok.decode(gen, skip_special_tokens=True).strip()
    return text

## 11) Indexing & Chat helpers for the UI

In [14]:
# Global state for a single-session demo
_vs: Optional[VSHandle] = None
_embedder_cfg: Optional[EmbedderCfg] = None
_embedder = None
_persist_dir = DEFAULT_PERSIST_DIR
_collection_name = "personal_kb"
_store_kind = "chroma"

def scan_folder(base_dir: str) -> List[str]:
    paths = []
    for ext in SUPPORTED_SUFFIXES:
        paths.extend(glob.glob(os.path.join(base_dir, f"**/*{ext}"), recursive=True))
    return sorted(list(set(paths)))

def ingest(base_dir: str, uploaded_files, chunk_size: int, chunk_overlap: int, store_kind: str, persist_dir: str, collection_name: str, embedder_kind: str, embedder_model: str):
    global _vs, _embedder_cfg, _embedder, _persist_dir, _collection_name, _store_kind
    _persist_dir = persist_dir or DEFAULT_PERSIST_DIR
    _collection_name = collection_name or "personal_kb"
    _store_kind = store_kind

    # 1) pick embedder
    kind_map = {
        "OpenAI (text-embedding-3-small)": ("openai", "text-embedding-3-small"),
        "Local (all-MiniLM-L6-v2)": ("local-minilm", "sentence-transformers/all-MiniLM-L6-v2"),
        "Local (bge-small-en-v1.5)": ("local-bge", "BAAI/bge-small-en-v1.5"),
    }
    if embedder_kind in kind_map:
        e_kind, e_model = kind_map[embedder_kind]
    else:
        e_kind, e_model = embedder_kind, embedder_model  # advanced: custom string

    _embedder_cfg = EmbedderCfg(kind=e_kind, model=e_model)
    _embedder = create_embedder(_embedder_cfg)

    # 2) collect file paths
    files = []
    if base_dir and os.path.isdir(base_dir):
        files.extend(scan_folder(base_dir))
    # Save uploaded files to a temp dir
    up_dir = tempfile.mkdtemp(prefix="uploads_")
    if uploaded_files:
        for uf in uploaded_files:
            # gradio File may be tmp path or NamedString
            src = uf.name if hasattr(uf, "name") else str(uf)
            if os.path.isfile(src):
                dst = os.path.join(up_dir, os.path.basename(src))
                shutil.copyfile(src, dst)
                files.append(dst)
    files = [p for p in files if os.path.splitext(p)[1].lower() in SUPPORTED_SUFFIXES]
    if not files:
        return "⚠️ No supported files found.", None

    # 3) load texts into docs
    docs = []
    for p in sorted(set(files)):
        txt = read_any(p)
        if not txt:
            continue
        docs.append({"text": txt, "metadata": {"source": p}})

    if not docs:
        return "⚠️ No text could be extracted from provided files.", None

    # 4) chunk
    chunks = chunk_documents(docs, chunk_size=chunk_size, chunk_overlap=chunk_overlap)

    # 5) build vector store
    _vs = build_vector_store(chunks, _embedder, kind=store_kind, persist_dir=_persist_dir, collection_name=_collection_name)
    if store_kind == "chroma":
        # Ensure persistence
        _vs.handle.persist()

    return f"✅ Indexed {len(files)} files into {store_kind.upper()} with {len(chunks)} chunks.", len(chunks)

def clear_index(store_kind: str, persist_dir: str):
    global _vs
    if store_kind == "chroma":
        try:
            if os.path.isdir(persist_dir):
                shutil.rmtree(persist_dir)
            os.makedirs(persist_dir, exist_ok=True)
        except Exception as e:
            return f"⚠️ Failed to clear Chroma dir: {e}"
    _vs = None
    return "✅ Cleared index."

def ask(question: str, top_k: int, model_path: str, use_frontier: bool, frontier_model: str, temperature: float, max_tokens: int, show_prompt: bool):
    if not question or not question.strip():
        return None, "⚠️ Enter a question.", None, None
    if _vs is None:
        return None, "⚠️ Please index documents first.", None, None
    docs = retrieve_docs(_vs, question, k=int(top_k))
    ctx, used = build_context_str(docs)
    prompt_preview = f"{SYSTEM_PROMPT}\n\nContext (top {top_k}):\n{ctx}\n\nQuestion: {question}"
    try:
        if use_frontier:
            ans = generate_frontier_answer(question, ctx, model=frontier_model, temperature=float(temperature), max_tokens=int(max_tokens))
        else:
            ans = generate_local_answer(question, ctx, model_id=model_path, temperature=float(temperature), max_new_tokens=int(max_tokens))
    except Exception as e:
        return None, f"⚠️ Generation error: {e}", prompt_preview if show_prompt else None, used
    return ans, None, (prompt_preview if show_prompt else None), used

## 12) Gradio App

In [15]:
def format_sources(used):
    if not used:
        return ""
    lines = []
    for u in used:
        src = u.get("source", "unknown")
        snippet = (u.get("snippet","")[:300] + "...") if len(u.get("snippet",""))>300 else u.get("snippet","")
        lines.append(f"- **{os.path.basename(src)}** — {snippet}")
    return "\n".join(lines)

with gr.Blocks(title="Personal AI Knowledge Worker (RAG)") as app:
    gr.Markdown("## Personal AI Knowledge Worker — RAG over your files")
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1) Index your files")
            base_dir = gr.Textbox(label="Base folder (e.g., /content/drive/MyDrive/KnowledgeBase)", value="")
            uploads = gr.Files(label="Or upload files", type="filepath")
            chunk_size = gr.Slider(256, 2000, value=800, step=16, label="Chunk size")
            chunk_overlap = gr.Slider(0, 600, value=200, step=10, label="Chunk overlap")
            store_kind = gr.Radio(label="Vector store", choices=["chroma","faiss"], value="chroma")
            persist_dir = gr.Textbox(label="Chroma persist directory", value=DEFAULT_PERSIST_DIR)
            collection_name = gr.Textbox(label="Collection name", value="personal_kb")
            # Default to OpenAI embeddings (like day5.ipynb) to avoid cache issues
            embedder_kind = gr.Dropdown(label="Embeddings", value="OpenAI (text-embedding-3-small)", choices=[
                "OpenAI (text-embedding-3-small)",
                "Local (all-MiniLM-L6-v2)",
                "Local (bge-small-en-v1.5)",
            ])
            embedder_model = gr.Textbox(label="(Advanced) Custom embedding model id", value="")
            btn_index = gr.Button("Index Documents", variant="primary")
            index_status = gr.Textbox(label="Index status")
            num_chunks = gr.Number(label="# Chunks", value=None)
            btn_clear = gr.Button("Clear Index")
            clear_status = gr.Textbox(label="Clear status")
        with gr.Column(scale=1):
            gr.Markdown("### 2) Chat with your Knowledge Base")
            use_frontier = gr.Checkbox(label="Use Frontier (OpenAI) instead of local LLM", value=False)
            frontier_model = gr.Textbox(label="Frontier model (OpenAI)", value="gpt-4o-mini")
            local_model = gr.Textbox(label="Local LLM (HF)", value=DEFAULT_LOCAL_LLM)
            temperature = gr.Slider(0.0, 1.5, value=0.2, step=0.05, label="Temperature")
            max_tokens = gr.Slider(64, 2048, value=512, step=32, label="Max new tokens")
            top_k = gr.Slider(1, 30, value=8, step=1, label="Top‑K chunks")
            show_prompt = gr.Checkbox(label="Show prompt preview & contexts", value=True)
            question = gr.Textbox(label="Your question", value="", lines=3)
            btn_ask = gr.Button("Ask", variant="primary")
            answer = gr.Markdown(label="Answer")
            warn = gr.Textbox(label="Warnings/Errors")
            prompt_prev = gr.Textbox(label="Prompt preview", lines=10)
            sources_md = gr.Markdown(label="Sources used")

    btn_index.click(
        fn=ingest,
        inputs=[base_dir, uploads, chunk_size, chunk_overlap, store_kind, persist_dir, collection_name, embedder_kind, embedder_model],
        outputs=[index_status, num_chunks]
    )
    btn_clear.click(
        fn=clear_index,
        inputs=[store_kind, persist_dir],
        outputs=[clear_status]
    )
    def _ask_and_format(q, k, lm_path, use_f, fm, temp, mx, showp):
        ans, err, prompt, used = ask(q, k, lm_path, use_f, fm, temp, mx, showp)
        srcs = format_sources(used)
        return ans or "", err or "", prompt or "", srcs or ""
    btn_ask.click(
        fn=_ask_and_format,
        inputs=[question, top_k, local_model, use_frontier, frontier_model, temperature, max_tokens, show_prompt],
        outputs=[answer, warn, prompt_prev, sources_md]
    )

print("✅ UI ready. In Colab/Jupyter, run: app.queue().launch(share=True)")
print("💡 Note: UI defaults to OpenAI embeddings to avoid cache corruption issues")

✅ UI ready. In Colab/Jupyter, run: app.queue().launch(share=True)
💡 Note: UI defaults to OpenAI embeddings to avoid cache corruption issues


### 13) Launch the app

In [ ]:
# In Colab, use share=True for an external link:
import gradio as gr
gr.close_all()
app.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7cdba496fba6996387.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_20873/343304617.py:71: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  _vs.handle.persist()
/tmp/ipykernel_20873/1601265530.py:28: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)


---
## Notes & suggestions
- **T4 GPU**: The default local LLM (`Phi-3-mini-4k-instruct`) loads in **4‑bit** and should fit on a **T4‑16GB**. If you hit OOM, try a smaller model or reduce `max_tokens`.
- **Vector stores**: Use **Chroma** if you want your index to persist across runs; **FAISS** is fast and can be saved/loaded on demand.
- **Troubleshooting retrieval** (from Day 126): if answers are missing, try increasing **Top‑K**, reducing **chunk size**, or increasing **chunk overlap** so relevant context is more likely to be included.
- **Privacy**: To avoid sending private data to external APIs, select **local embeddings** and **local LLM**.
- **File types**: This notebook uses lightweight loaders; for complex Office/PDFs, consider the `unstructured` library (heavier) or domain‑specific parsers.